# FitzHugh-Nagumo with a random on/off input

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/27_telegraph_fhn_markov.ipynb)

The input switches on and off at random: `s` is a two-state Markov variable with rates
`kon` (off → on) and `koff` (on → off). With the input on the cell oscillates; off, it
rests.

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.4 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.5.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("telegraph_fhn")
      .markov("s", [["0", "kon"], ["koff", "0"]])
      .ode("v", "v-v^3/3-w+i0+di*s")
      .ode("w", "eps*(v+a-b*w)")
      .aux("input", "i0+di*s")
      .parameter(i0=0, di=0.8, kon=0.02, koff=0.02, eps=0.08, a=0.7, b=0.8)
      .initial(v=-1.2, w=-0.62, s=0)
      .option(total=600, dt=0.05))
print(mb.render())

markov s 2
{0} {kon}
{koff} {0}
v'=v-v^3/3-w+i0+di*s
w'=eps*(v+a-b*w)
aux input=i0+di*s
par i0=0,di=0.8,kon=0.02,koff=0.02,eps=0.08,a=0.7,b=0.8
init v=-1.2,w=-0.62,s=0
@ total=600,dt=0.05
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="v", y="w", window=(-2.5, 2.5, -1, 2), df=False, nullclines=False,
              sliders={"kon": (0, 0.2), "koff": (0, 0.2), "di": (0, 1.5)}, show=["v", "input"],
              title="Telegraph-driven FitzHugh-Nagumo")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Fraction of time the input is on**, against the theory kon/(kon + koff).

In [5]:
tr = ex.m.run(keep=False, total=20000)
print("fraction on:", round(float(tr["s"].mean()), 3), " theory:", round(ex.m.pars["kon"] / (ex.m.pars["kon"] + ex.m.pars["koff"]), 3))

fraction on: 0.515  theory: 0.5


## Things to try

- Make `koff` larger than `kon`: mostly resting with occasional bursts.